# Mikołaj: RAG (Część II)

Zakres (3 pkt): Część II z [instrukcji](../Zadanie_3_zmodyfikowane.md). Ta część **nie zależy od Części I**: na start wystarczy własny indeks FAISS. Retriever hybrydowy z Części I (pkt 2.b) można podpiąć później z `../utils/`.

## Do zrobienia

1. **II.A: artykuł o RAG** (Medium, RAG basics). Krótka notatka w notebooku.
2. **Pkt 1: API CLARIN**: rejestracja na [services.clarin-pl.eu](https://services.clarin-pl.eu/login), lista modeli (`/api/v1/oapi/models`), uruchomienie modelu (`/api/v1/oapi/chat/completions`). Klucz API trzymać poza repo (zmienna środowiskowa).
3. **Pkt 2: dane i indeks**: wybrać zbiór (np. IMDB `stanfordnlp/imdb`) i zaindeksować go (lub użyć gotowego indeksu FAISS Wikipedii z Kaggle).
4. **Pkt 3: moduł generacji (RAG)**:
   - a. prompt zamieniający znalezione dokumenty na odpowiedź w języku naturalnym,
   - b. własne przykładowe zapytania przepuszczone przez retriever i moduł RAG,
   - c. analiza rezultatów po zastosowaniu RAG,
   - d. podsumowanie wyników.

In [1]:
import sys

sys.path.insert(0, "..")

## II.A: notatka z artykułu

RAG (Retrieval-Augmented Generation) łączy dwa elementy: wyszukiwarkę (retriever) i model językowy (generator). Najpierw dokumenty dzieli się na fragmenty, zamienia na wektory (osadzenia) i zapisuje w bazie wektorowej, np. FAISS. Przy pytaniu retriever zamienia je na wektor i znajduje najbardziej podobne fragmenty. Te fragmenty trafiają do promptu razem z pytaniem, a model odpowiada na ich podstawie. Po co: model dostaje aktualną lub prywatną wiedzę spoza swoich wag, rzadziej zmyśla (halucynuje) i można wskazać źródło odpowiedzi. Prompt jest kluczowy, bo mówi modelowi, żeby używał tylko podanego kontekstu. Ograniczenia: jakość odpowiedzi zależy od retrievera (zły fragment daje złą odpowiedź), a liczba fragmentów jest ograniczona długością kontekstu modelu.

## 1. Model językowy CLARIN

Token jest w `03/.env` (wzór: `03/.env.example`), czytany przez `pydantic-settings`.

In [2]:
import random

import pandas as pd

from utils.rag.data import load_squad
from utils.rag.retriever import FaissRetriever
from utils.rag.llm import list_models, get_llm
from utils.rag.pipeline import build_rag_chain, build_plain_chain, contains_gold

pd.set_option("display.max_colwidth", None)
list_models()

['agent-worker-example',
 'dariah-eventstotriples',
 'gemma-4-26b-a4b-it',
 'gemma-4-31b-it',
 'gpt-oss-20b',
 'llama',
 'llama',
 'llama3.1-8b',
 'pllum',
 'pllum',
 'pllum-12b-chat',
 'pllum-fusion']

**Wybór modelu:** `gemma-4-31b-it` (Google Gemma 4, 31 mld parametrów, wersja dostrojona do instrukcji). To największy ogólny model czatowy na liście i dobrze radzi sobie z angielskim, w którym jest SQuAD. `pllum` jest nastawiony na polski, a `llama3.1-8b` jest znacznie mniejszy. Nazwa jest w `CLARIN_MODEL` w `03/.env`.

In [3]:
llm = get_llm()
llm.invoke("Say hello in Polish.").content

10:33:57 | INFO    | utils.rag.llm | LLM: gemma-4-31b-it (temperature=0.0)


'Cześć! (Informal)\nDzień dobry! (Formal/Good morning)'

## 2. Indeks i wyszukiwanie

Akapity SQuAD (split `validation`) osadzone modelem `multi-qa-mpnet-base-dot-v1`, indeks FAISS z iloczynem skalarnym. Osadzenia są zapisywane w `03/artifacts/rag/`, więc kolejne uruchomienie jest szybkie.

In [4]:
docs, qa = load_squad()
retriever = FaissRetriever(docs)

10:34:01 | INFO    | utils.rag.data | SQuAD validation: 2067 unique paragraphs, 10570 questions


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

10:34:03 | INFO    | utils.rag.retriever | Loaded 2067 cached embeddings from /Users/mikolajmachalski/Studia/2sem/NLP/03/artifacts/rag/squad_validation__multi-qa-mpnet-base-dot-v1.npy


In [5]:
search_queries = [
    "Who won Super Bowl 50?",
    "What is the capital of Poland?",
    "How do plants make energy?",
]
pd.DataFrame(
    [
        {"query": q, "rank": r, "score": round(d.score, 1), "title": d.title, "text": d.text[:100]}
        for q in search_queries
        for r, d in enumerate(retriever.retrieve(q, k=3), 1)
    ]
)

,query,rank,score,title,text
0,Who won Super Bowl 50?,1,24.3,Super Bowl 50,Super Bowl 50 was an American football game to determine the champion of the National Football Leagu
1,Who won Super Bowl 50?,2,21.6,Super Bowl 50,Super Bowl 50 featured numerous records from individuals and teams. Denver won despite being massive
2,Who won Super Bowl 50?,3,20.8,Super Bowl 50,The Broncos took an early lead in Super Bowl 50 and never trailed. Newton was limited by Denver's de
3,What is the capital of Poland?,1,23.6,Warsaw,Warsaw (Polish: Warszawa [varˈʂava] ( listen); see also other names) is the capital and largest city
4,What is the capital of Poland?,2,21.3,Warsaw,"Warsaw remained the capital of the Polish–Lithuanian Commonwealth until 1796, when it was annexed by"
5,What is the capital of Poland?,3,21.3,Warsaw,"The first historical reference to Warsaw dates back to the year 1313, at a time when Kraków served a"
6,How do plants make energy?,1,19.0,Chloroplast,"Chloroplasts' main role is to conduct photosynthesis, where the photosynthetic pigment chlorophyll c"
7,How do plants make energy?,2,19.0,Chloroplast,"Like mitochondria, chloroplasts use the potential energy stored in an H+, or hydrogen ion gradient t"
8,How do plants make energy?,3,18.6,Chloroplast,Photorespiration can occur when the oxygen concentration is too high. Rubisco cannot distinguish bet


Wszystkie trzy wyszukiwania trafiają we właściwe artykuły (Super Bowl 50, Warsaw, Chloroplast). Pytanie o rośliny jest sformułowane potocznie, a najbliżej są akapity o fotosyntezie w chloroplastach: wyszukiwanie gęste łapie znaczenie, nie tylko wspólne słowa. Wyniki (iloczyn skalarny) nie mają stałej skali, więc służą tylko do porównania akapitów dla jednego pytania.

## 3. Łańcuchy RAG i bez RAG

`rag_chain`: pytanie → retriever (`k = 5`) → prompt z numerowanymi akapitami → model → `{question, docs, answer}`. `plain_chain`: samo pytanie do modelu, bez kontekstu.

In [6]:
rag_chain = build_rag_chain(retriever, llm, k=5)
plain_chain = build_plain_chain(llm)

In [7]:
# Article titles in the index, to check which own questions are covered by the base.
sorted({d.title for d in docs})

['1973 oil crisis',
 'Amazon rainforest',
 'American Broadcasting Company',
 'Apollo program',
 'Black Death',
 'Chloroplast',
 'Civil disobedience',
 'Computational complexity theory',
 'Construction',
 'Ctenophora',
 'Doctor Who',
 'Economic inequality',
 'European Union law',
 'Force',
 'French and Indian War',
 'Fresno, California',
 'Genghis Khan',
 'Geology',
 'Harvard University',
 'Huguenot',
 'Immune system',
 'Imperialism',
 'Intergovernmental Panel on Climate Change',
 'Islamism',
 'Jacksonville, Florida',
 'Kenya',
 'Martin Luther',
 'Newcastle upon Tyne',
 'Nikola Tesla',
 'Normans',
 'Oxygen',
 'Packet switching',
 'Pharmacy',
 'Prime number',
 'Private school',
 'Rhine',
 'Scottish Parliament',
 'Sky (United Kingdom)',
 'Southern California',
 'Steam engine',
 'Super Bowl 50',
 'Teacher',
 'United Methodist Church',
 'University of Chicago',
 'Victoria (Australia)',
 'Victoria and Albert Museum',
 'Warsaw',
 'Yuan dynasty']

Po obejrzeniu listy tytułów **dostosować pytania własne** tak, żeby dwa były z tematów obecnych w bazie, a dwa spoza bazy (wtedy oczekiwane jest `I don't know.`). Pytanie z fałszywym założeniem sprawdza, czy model da się podpuścić.

In [8]:
questions = [
    # from SQuAD
    qa[0].question,
    qa[100].question,
    qa[2000].question,
    # own, topic in the base
    "Which city hosted Super Bowl 50?",
    "What did Martin Luther publish in 1517?",
    # own, topic outside the base
    "What is the population of Wrocław?",
    "Who wrote the novel Pan Tadeusz?",
    # false premise
    "Who invented the steam engine in Poland?",
]
rows = []
for q in questions:
    out = rag_chain.invoke(q)
    rows.append(
        {
            "question": q,
            "plain": plain_chain.invoke({"question": q}),
            "rag": out["answer"],
            "sources": [d.title for d in out["docs"]],
        }
    )
pd.DataFrame(rows)

,question,plain,rag,sources
0,Which NFL team represented the AFC at Super Bowl 50?,The Denver Broncos represented the AFC at Super Bowl 50.,The Denver Broncos represented the AFC at Super Bowl 50 [1].,"[Super Bowl 50, Super Bowl 50, Super Bowl 50, Super Bowl 50, Super Bowl 50]"
1,Who were special guests for the Super Bowl halftime show?,"Many artists have been special guests over the years, including Beyoncé and Bruno Mars during Katy Perry's performance, and Drake and Nicki Minaj during Rihanna's show.",The special guest performers were Beyoncé and Bruno Mars [2].,"[Super Bowl 50, Super Bowl 50, Super Bowl 50, Super Bowl 50, Super Bowl 50]"
2,What is dramatic gesturing an example of?,Dramatic gesturing is an example of nonverbal communication.,Dramatic gesturing is an example of a nonverbal expression of enthusiasm [1].,"[Teacher, Martin Luther, Computational complexity theory, Yuan dynasty, Teacher]"
3,Which city hosted Super Bowl 50?,"Super Bowl 50 was hosted in Santa Clara, California.","Super Bowl 50 was played at Levi's Stadium in Santa Clara, California [2].","[Super Bowl 50, Super Bowl 50, Super Bowl 50, Super Bowl 50, Super Bowl 50]"
4,What did Martin Luther publish in 1517?,Martin Luther published his *Ninety-five Theses* in 1517.,"In 1517, Martin Luther proposed an academic discussion on the power and usefulness of indulgences in his Ninety-Five Theses [1].","[Martin Luther, Martin Luther, Martin Luther, Martin Luther, Martin Luther]"
5,What is the population of Wrocław?,"The population of Wrocław is approximately 670,000 people.",I don't know.,"[Warsaw, Warsaw, Warsaw, Warsaw, Warsaw]"
6,Who wrote the novel Pan Tadeusz?,Adam Mickiewicz wrote the novel *Pan Tadeusz*.,I don't know.,"[Warsaw, Pharmacy, Martin Luther, Victoria and Albert Museum, Victoria and Albert Museum]"
7,Who invented the steam engine in Poland?,There is no record of a Polish inventor who created the steam engine; it was developed primarily by Thomas Newcomen and James Watt in Britain.,I don't know.,"[Steam engine, Steam engine, Steam engine, Steam engine, Steam engine]"


Obserwacje z tabeli:

- **Pytania zależne od kontekstu.** „Who were special guests for the Super Bowl halftime show?” bez RAG daje ogólną listę z różnych lat, a z RAG konkretną odpowiedź dla Super Bowl 50 (Beyoncé i Bruno Mars) ze wskazaniem akapitu. Pytania SQuAD często mają sens tylko przy konkretnym akapicie.
- **Pytania spoza bazy** (Wrocław, Pan Tadeusz): retriever zwraca najbliższe, ale niezwiązane akapity (np. Warsaw), a RAG odpowiada `I don't know.`. Model bez RAG odpowiada z pamięci: o Pan Tadeuszu poprawnie (choć nie poprawia, że to poemat, a nie powieść), a liczba mieszkańców Wrocławia jest nie do sprawdzenia w bazie.
- **Fałszywe założenie** (silnik parowy w Polsce): RAG znajduje akapity o silniku parowym, nie widzi w nich odpowiedzi i mówi `I don't know.`. Model bez RAG sam prostuje założenie.
- Przy pytaniach z bazy obie wersje odpowiadają poprawnie, ale RAG podaje źródło `[n]`.

## 4. Miara liczbowa (50 pytań)

Losowa próbka 50 pytań z SQuAD (ziarno 42). Cztery liczby:

- trafność retrievera: czy akapit z odpowiedzią jest w top 5 (hit@5),
- RAG: czy odpowiedź zawiera wzorzec (`contains_gold`),
- bez RAG: to samo dla modelu bez kontekstu,
- liczba odpowiedzi `I don't know.` z RAG.

CLARIN ogranicza liczbę równoległych zapytań (HTTP 429), więc zapytania idą po kolei z ponawianiem (`with_retry`).

In [10]:
N = 50
sample = random.Random(42).sample(qa, N)
qs = [s.question for s in sample]
# CLARIN rate-limits parallel requests (HTTP 429): go one by one and retry with backoff.
batch_config = {"max_concurrency": 1}
rag_out = rag_chain.with_retry(stop_after_attempt=6).batch(qs, config=batch_config)
plain_out = plain_chain.with_retry(stop_after_attempt=6).batch(
    [{"question": q} for q in qs], config=batch_config
)

counts = {
    "retriever hit@5": sum(s.doc_id in {d.id for d in o["docs"]} for s, o in zip(sample, rag_out)),
    "RAG contains gold": sum(
        contains_gold(o["answer"], s.answers) for s, o in zip(sample, rag_out)
    ),
    "no-RAG contains gold": sum(contains_gold(p, s.answers) for s, p in zip(sample, plain_out)),
    'RAG "I don\'t know."': sum(o["answer"].strip() == "I don't know." for o in rag_out),
}
pd.DataFrame({"count": counts, "share": {k: v / N for k, v in counts.items()}})

,count,share
retriever hit@5,46,0.92
RAG contains gold,44,0.88
no-RAG contains gold,16,0.32
"RAG ""I don't know.""",3,0.06


## 5. Przykłady błędów

Pierwsze 6 pytań, na które odpowiedź RAG nie zawiera wzorca, razem z informacją, czy właściwy akapit został znaleziony.

In [11]:
by_id = {d.id: d for d in docs}
misses = [(s, o) for s, o in zip(sample, rag_out) if not contains_gold(o["answer"], s.answers)]
pd.DataFrame(
    [
        {
            "question": s.question,
            "gold": s.answers,
            "rag": o["answer"],
            "right paragraph retrieved": s.doc_id in {d.id for d in o["docs"]},
            "right article": by_id[s.doc_id].title,
            "retrieved": [d.title for d in o["docs"]],
        }
        for s, o in misses[:6]
    ]
)

,question,gold,rag,right paragraph retrieved,right article,retrieved
0,In what city is SAP Center located?,"[San Jose, San Jose, San Jose.]",I don't know.,False,Super Bowl 50,"[Warsaw, University of Chicago, Southern California, Southern California, Apollo program]"
1,"For what invention was U.S. Patent 1,655,114 granted?","[VTOL aircraft, VTOL aircraft, a biplane capable of taking off vertically]",I don't know.,False,Nikola Tesla,"[Nikola Tesla, Nikola Tesla, Nikola Tesla, Nikola Tesla, Nikola Tesla]"
2,What does Obersee mean?,"[upper lake, upper lake, (""upper lake"", upper lake]",I don't know.,False,Rhine,"[Imperialism, Imperialism, Newcastle upon Tyne, Martin Luther, Huguenot]"
3,"What interpretation of Islam is, for many of the adherents, the ""gold standard"" of their religion?","[Saudi, the Saudi-interpretation, Saudi]","The Saudi-based Wahhabist interpretation of Islam has been set as the ""gold standard"" of religion in the minds of some or many Muslims [1].",True,Islamism,"[Islamism, Islamism, Islamism, Islamism, Islamism]"
4,What chemical element was the cause of the Apollo 1 disastrous outcome?,[pure O],The disaster occurred because the capsule was pressurized with pure oxygen [3].,True,Oxygen,"[Apollo program, Apollo program, Oxygen, Apollo program, Apollo program]"
5,Where are health and social problems most common?,"[among states in the US with larger income inequalities, countries with bigger income inequalities, countries with bigger income inequalities]","Health and social problems are more common in countries and US states with larger income inequalities [1], [2].",True,Economic inequality,"[Economic inequality, Economic inequality, Economic inequality, Economic inequality, Economic inequality]"


## Analiza

Wyniki na 50 pytaniach: retriever hit@5 = 46/50 (92%), RAG zawiera wzorzec 44/50 (88%), bez RAG 16/50 (32%), `I don't know.` 3/50.

Sześć pytań, w których odpowiedź RAG nie zawiera wzorca, dzielę na cztery kategorie:

- **(a) Retriever nie znalazł akapitu: 3.** SAP Center, patent Tesli, „Obersee”. We wszystkich trzech model uczciwie odpowiada `I don't know.` zamiast zgadywać. Przy patencie retriever trafił we właściwy artykuł (Nikola Tesla), ale nie we właściwy akapit.
- **(b) Akapit był, odpowiedź poprawna, ale miara jej nie uznała: 3.** „Saudi-based Wahhabist interpretation” (normalizacja usuwa myślnik i skleja „saudibased”, więc wzorzec „saudi” nie pasuje), „pure oxygen” wobec dziwnego wzorca „pure O”, oraz poprawna parafraza o nierównościach dochodowych. To błędy miary, nie modelu.
- **(c) Halucynacja: 0.** Nie znalazłem odpowiedzi bez pokrycia w kontekście.
- **(d) `I don't know.` mimo dobrego akapitu: 0.** Wszystkie trzy odmowy wynikają z błędu retrievera.

Po ręcznej korekcie kategorii (b) RAG odpowiada poprawnie na 47/50 pytań, a wszystkie pozostałe błędy to odmowy przy złym kontekście. Model nie zmyśla, gdy kontekst jest zły.

Pytania spoza bazy: RAG odmawia odpowiedzi, model bez RAG odpowiada z pamięci (czasem dobrze, ale nie da się tego sprawdzić w bazie).

Porównanie z modelem bez RAG: różnica 88% wobec 32%. Część tej różnicy to pytania, które bez akapitu są niejednoznaczne (np. który halftime show), a część to sformułowania inne niż wzorzec. Miara jest surowa dla obu wersji, ale odpowiedzi RAG są krótsze i bliższe słownictwu akapitu, więc trafiają we wzorzec częściej.

## Podsumowanie

RAG z modelem `gemma-4-31b-it` podnosi odsetek odpowiedzi zawierających wzorzec z 32% do 88% (po ręcznej korekcie błędów miary: 94%). Górną granicę wyznacza retriever (92% hit@5): gdy nie znajdzie akapitu, model odpowiada `I don't know.`, więc prompt skutecznie ogranicza halucynacje. Poza bazą RAG odmawia odpowiedzi, a model bez RAG odpowiada z pamięci bez źródła. Największy zysk dałby lepszy retriever, np. hybrydowy z Części I (BM25 pomaga przy rzadkich nazwach jak „Obersee” czy numer patentu) albo większe `k`, oraz łagodniejsza miara (np. dopasowanie tokenów zamiast podciągu).